In [1]:
%load_ext kedro.ipython

The kedro.ipython extension is already loaded. To reload it, use:
  %reload_ext kedro.ipython


In [2]:
!uv pip install --upgrade seaborn matplotlib

Using Python 3.12.7 environment at: C:\Users\arczi\source\repos\Mini\Msi\wsi-reg-cls\wsi-reg\.venv
Resolved 14 packages in 159ms
Prepared 5 packages in 2ms
Uninstalled 5 packages in 882ms
Installed 5 packages in 896ms
 - fonttools==4.61.1
 + fonttools==4.62.0
 - kiwisolver==1.4.9
 + kiwisolver==1.5.0
 - numpy==2.4.2
 + numpy==2.4.3
 - pandas==2.3.3
 + pandas==3.0.1
 - seaborn==0.12.2
 + seaborn==0.13.2


In [3]:
import pandas as pd

df = catalog.load("domy")

df = df.sort_index(axis=1)

[03/12/26 23:15:06] INFO     Loading data from domy (CSVDataset)...                            ]8;id=779970;file://C:\Users\arczi\source\repos\Mini\Msi\wsi-reg-cls\wsi-reg\.venv\Lib\site-packages\kedro\io\data_catalog.py\data_catalog.py]8;;\:]8;id=506313;file://C:\Users\arczi\source\repos\Mini\Msi\wsi-reg-cls\wsi-reg\.venv\Lib\site-packages\kedro\io\data_catalog.py#1048\1048]8;;\

In [6]:
df['Alley'] = df['Alley'].replace('?', 'NoAlley')

In [7]:
import numpy as np

kolumny_piwnica = ['BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2', 'BsmtQual']

for kol in kolumny_piwnica:
    df[kol] = df[kol].replace('?', np.nan)

maska_brak_piwnicy = df['BsmtQual'].isna()

for kol in kolumny_piwnica:
    df.loc[maska_brak_piwnicy, kol] = 0

for kol in ['BsmtExposure', 'BsmtFinType2']:
    najczestsza = df[kol].mode()[0]
    df[kol] = df[kol].fillna(najczestsza)

In [8]:
kol = 'Electrical'
zmiana = df[kol].mode()[0]
df[kol] = df[kol].replace('?', zmiana)

In [9]:
kol = 'Fence'
zmiana = 'NoFence'
df[kol] = df[kol].replace('?', zmiana)

In [10]:
df.loc[df['Fireplaces'] == 0, 'FireplaceQu'] = 'None'

In [13]:
import pandas as pd

garage_categorical = ['GarageType', 'GarageFinish', 'GarageQual', 'GarageCond']

for col in garage_categorical:
    df[col] = df[col].replace('?', 'None')

df['GarageYrBlt'] = df['GarageYrBlt'].replace('?', 0)
df['GarageYrBlt'] = pd.to_numeric(df['GarageYrBlt'])

In [14]:
import numpy as np

df['LotFrontage'] = df['LotFrontage'].replace('?', np.nan)
df['LotFrontage'] = pd.to_numeric(df['LotFrontage'])

df["LotFrontage"] = df.groupby("Neighborhood")["LotFrontage"].transform(
    lambda x: x.fillna(x.median())
)

In [35]:
most_frequent_type = df['MasVnrType'].mode()[0]

df.loc[(df['MasVnrType'].isnull()) & ((df['MasVnrArea'] == 0) | (df['MasVnrArea'] == 1)), 'MasVnrType'] = 'None'
df['MasVnrType'] = df['MasVnrType'].replace('?', 'None')
df['MasVnrArea'] = df['MasVnrArea'].replace('?', 0)
df['MasVnrArea'] = pd.to_numeric(df['MasVnrArea'])
df['MasVnrType'] = df['MasVnrType'].fillna(most_frequent_type)

In [30]:
kol = 'MiscFeature'
zmiana = 'None'
df[kol] = df[kol].replace('?', zmiana)

In [31]:
kol = 'PoolQC'
zmiana = 'None'
df[kol] = df[kol].replace('?', zmiana)

In [32]:
zliczone_pytajniki = (df == '?').sum()

# 2. Wyświetlamy tylko te kolumny, w których wynik jest większy od zera
tylko_z_pytajnikiem = zliczone_pytajniki[zliczone_pytajniki > 0]

print(tylko_z_pytajnikiem)

Series([], dtype: int64)


In [33]:
braki = df.isna().sum()

# Wyświetlamy tylko te kolumny, gdzie liczba braków jest > 0
print(braki[braki > 0])

Series([], dtype: int64)


In [36]:
with pd.option_context('display.max_rows', None):
    print(df.dtypes)

1stFlrSF           int64
2ndFlrSF           int64
3SsnPorch          int64
Alley             object
BedroomAbvGr       int64
BldgType          object
BsmtCond          object
BsmtExposure      object
BsmtFinSF1         int64
BsmtFinSF2         int64
BsmtFinType1      object
BsmtFinType2      object
BsmtFullBath       int64
BsmtHalfBath       int64
BsmtQual          object
BsmtUnfSF          int64
CentralAir        object
Condition1        object
Condition2        object
Electrical        object
EnclosedPorch      int64
ExterCond         object
ExterQual         object
Exterior1st       object
Exterior2nd       object
Fence             object
FireplaceQu       object
Fireplaces         int64
Foundation        object
FullBath           int64
Functional        object
GarageArea         int64
GarageCars         int64
GarageCond        object
GarageFinish      object
GarageQual        object
GarageType        object
GarageYrBlt      float64
GrLivArea          int64
HalfBath           int64


In [39]:
pd.set_option('display.max_columns', None)
df.select_dtypes(include=['object'])

,Alley,BldgType,BsmtCond,BsmtExposure,BsmtFinType1,BsmtFinType2,BsmtQual,CentralAir,Condition1,Condition2,Electrical,ExterCond,ExterQual,Exterior1st,Exterior2nd,Fence,FireplaceQu,Foundation,Functional,GarageCond,GarageFinish,GarageQual,GarageType,Heating,HeatingQC,HouseStyle,KitchenQual,LandContour,LandSlope,LotConfig,LotShape,MSZoning,MasVnrType,MiscFeature,Neighborhood,PavedDrive,PoolQC,RoofMatl,RoofStyle,SaleCondition,SaleType,Street,Utilities
0,NoAlley,1Fam,TA,No,GLQ,Unf,Gd,Y,Norm,Norm,SBrkr,TA,Gd,VinylSd,VinylSd,NoFence,None,PConc,Typ,TA,RFn,TA,Attchd,GasA,Ex,2Story,Gd,Lvl,Gtl,Inside,Reg,RL,BrkFace,None,CollgCr,Y,None,CompShg,Gable,Normal,WD,Pave,AllPub
1,NoAlley,1Fam,TA,Gd,ALQ,Unf,Gd,Y,Feedr,Norm,SBrkr,TA,TA,MetalSd,MetalSd,NoFence,TA,CBlock,Typ,TA,RFn,TA,Attchd,GasA,Ex,1Story,TA,Lvl,Gtl,FR2,Reg,RL,BrkFace,None,Veenker,Y,None,CompShg,Gable,Normal,WD,Pave,AllPub
2,NoAlley,1Fam,TA,Mn,GLQ,Unf,Gd,Y,Norm,Norm,SBrkr,TA,Gd,VinylSd,VinylSd,NoFence,TA,PConc,Typ,TA,RFn,TA,Attchd,GasA,Ex,2Story,Gd,Lvl,Gtl,Inside,IR1,RL,BrkFace,None,CollgCr,Y,None,CompShg,Gable,Normal,WD,Pave,AllPub
3,NoAlley,1Fam,Gd,No,ALQ,Unf,TA,Y,Norm,Norm,SBrkr,TA,TA,'Wd Sdng','Wd Shng',NoFence,Gd,BrkTil,Typ,TA,Unf,TA,Detchd,GasA,Gd,2Story,Gd,Lvl,Gtl,Corner,IR1,RL,BrkFace,None,Crawfor,Y,None,CompShg,Gable,Abnorml,WD,Pave,AllPub
4,NoAlley,1Fam,TA,Av,GLQ,Unf,Gd,Y,Norm,Norm,SBrkr,TA,Gd,VinylSd,VinylSd,NoFence,TA,PConc,Typ,TA,RFn,TA,Attchd,GasA,Ex,2Story,Gd,Lvl,Gtl,FR2,IR1,RL,BrkFace,None,NoRidge,Y,None,CompShg,Gable,Normal,WD,Pave,AllPub
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1455,NoAlley,1Fam,TA,No,Unf,Unf,Gd,Y,Norm,Norm,SBrkr,TA,TA,VinylSd,VinylSd,NoFence,TA,PConc,Typ,TA,RFn,TA,Attchd,GasA,Ex,2Story,TA,Lvl,Gtl,Inside,Reg,RL,BrkFace,None,Gilbert,Y,None,CompShg,Gable,Normal,WD,Pave,AllPub
1456,NoAlley,1Fam,TA,No,ALQ,Rec,Gd,Y,Norm,Norm,SBrkr,TA,TA,Plywood,Plywood,MnPrv,TA,CBlock,Min1,TA,Unf,TA,Attchd,GasA,TA,1Story,TA,Lvl,Gtl,Inside,Reg,RL,Stone,None,NWAmes,Y,None,CompShg,Gable,Normal,WD,Pave,AllPub
1457,NoAlley,1Fam,Gd,No,GLQ,Unf,TA,Y,Norm,Norm,SBrkr,Gd,Ex,CemntBd,CmentBd,GdPrv,Gd,Stone,Typ,TA,RFn,TA,Attchd,GasA,Ex,2Story,Gd,Lvl,Gtl,Inside,Reg,RL,BrkFace,Shed,Crawfor,Y,None,CompShg,Gable,Normal,WD,Pave,AllPub
1458,NoAlley,1Fam,TA,Mn,GLQ,Rec,TA,Y,Norm,Norm,FuseA,TA,TA,MetalSd,MetalSd,NoFence,None,CBlock,Typ,TA,Unf,TA,Attchd,GasA,Gd,1Story,Gd,Lvl,Gtl,Inside,Reg,RL,BrkFace,None,NAmes,Y,None,CompShg,Hip,Normal,WD,Pave,AllPub
